In [31]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
import joblib

In [32]:
df = pd.read_csv('insurance.csv')

In [33]:
df.duplicated().sum()

np.int64(0)

In [34]:
df.isna().sum()

Id               0
age              5
gender           0
bmi              0
bloodpressure    0
diabetic         0
children         0
smoker           0
region           3
claim            0
dtype: int64

In [35]:
df.dropna(inplace=True)

In [36]:
df.drop(columns=['Id'], inplace=True)

In [37]:
X = df[['age', 'gender', 'bmi', 'bloodpressure', 'diabetic', 'children', 'smoker']]
y = df[['claim']]

In [38]:
cat_cols = ['gender', 'diabetic', 'smoker']
label_encoders = {}

In [39]:
le = LabelEncoder()
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    label_encoders[col] = le
    
    joblib.dump(le, f"label_encoder_{col}.pkl")

C:\Users\noahg\AppData\Local\Temp\ipykernel_11816\1629543755.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = le.fit_transform(X[col])
C:\Users\noahg\AppData\Local\Temp\ipykernel_11816\1629543755.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = le.fit_transform(X[col])
C:\Users\noahg\AppData\Local\Temp\ipykernel_11816\1629543755.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

Se

In [42]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.30)

In [43]:
num_cols = ['age', 'bmi', 'bloodpressure', 'children']
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

In [44]:
joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']

In [45]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor


In [46]:
def evaluate_model(model, X_train, X_test, y_train, y_test):
    y_pred = model.predict(X_test)
    r2_score = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    return {"r2": r2_score, "MAE": mae, "RMSE": rmse}

In [47]:
results = {}

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
results['Linear Regression'] = evaluate_model(lr, X_train, X_test, y_train, y_test)
print("Linear Regression model trained!")

best_poly_model = None
best_poly_score = -np.inf

for degree in [2,3]:
    poly = PolynomialFeatures(degree=degree)
    X_train_poly = poly.fit_transform(X_train)
    X_test_poly = poly.transform(X_test)
    
    poly_lr = LinearRegression()
    poly_lr.fit(X_train_poly, y_train)
    
    score = poly_lr.score(X_test_poly, y_test)
    
    if score > best_poly_score:
        best_poly_score = score
        best_poly_model = (degree, poly, poly_lr)
        
degree, poly, poly_lr = best_poly_model

results[f'Polynomial Regression (deg = {degree})'] = evaluate_model(poly_lr, poly.fit_transform(X_train), poly.transform(X_test), y_train, y_test)
print("Polynomial Regression model trained!")

rf = RandomForestRegressor()

rf_params = {
    'n_estimators' : [100, 200, 500],
    "max_depth" : [None, 10, 20, 30],
    "min_samples_split" : [2, 5, 10],
    "min_samples_leaf" : [1, 2, 4]
}

rf_grid_obj = GridSearchCV(rf, rf_params, scoring='r2', cv=5, n_jobs=-1, verbose=2, refit=True )
rf_grid_obj.fit(X_train, y_train)
best_rf = rf_grid_obj.best_estimator_
best_rf_params = rf_grid_obj.best_params_

results['Random Forest'] = evaluate_model(best_rf, X_train, X_test, y_train, y_test)
print("Random Forest training is completed!, best parameters:", best_rf_params)

svr = SVR()

svr_params = {
    "kernel" : ['linear', 'rbf', 'poly'],
    "C" : [0.1, 1, 10, 100],
    "gamma" : ['scale', 'auto'],
    "epsilon" : [0.01, 0.1, 0.5],
    "degree" : [2,3]
}

svr_grid_obj = GridSearchCV(svr, svr_params, scoring='r2', n_jobs=-1, cv=5, verbose=2, refit=True)
svr_grid_obj.fit(X_train, y_train)
best_svr = svr_grid_obj.best_estimator_
best_svr_params = svr_grid_obj.best_params_


results['Support Vector Regressor'] = evaluate_model(best_svr, X_train, X_test, y_train, y_test)
print('Support Vector Regressor training is completed!, best parameters is:', best_svr_params)

   